# Winkansen van voetbal wedstrijden voorspellen
#### Een onderzoek van Lucas Hoetink, Hidde monsma, Khaleel Al-Aqel, Gamal Al-Sabaeei

In dit rapport gaan wij de winkansen van voetbalwedstrijden voorspellen. Dit gaan we doen op basis van principes uit de data science en doormiddel van machine learning modellen.

---

## Importing Libraries

Als eerste moeten we altijd de juiste libraries importeren zodat we toegang hebben tot de benodigde tools.

In [3]:
import sqlite3 as sql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import subprocess
import os
from pathlib import Path
import tempfile

---

## Loading Data

Als volgende moeten we de data ophalen en inladen uit de `SQL` database

In [4]:
subprocess.run(['pip', 'install', 'kaggle', '-q'], capture_output=True)

# Set up Kaggle credentials
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

credentials = {
    "username": "lucashoetink",
    "key": "22e34cd52b72413f58087cacec1636c7"
}

with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump(credentials, f)

os.chmod(kaggle_dir / 'kaggle.json', 0o600)

# Download to a writable temp directory
temp_dir = tempfile.gettempdir()
os.system(f'kaggle datasets download -d hugomathien/soccer --unzip -p {temp_dir}')

# Connect to the database from temp directory
db_path = os.path.join(temp_dir, 'database.sqlite')
connection = sql.connect(db_path)
print(f"Connected to database at: {db_path}")

Connected to database at: C:\Users\gamal\AppData\Local\Temp\database.sqlite


---

## SQL Exploration

In [5]:
# See all table names
cursor = connection.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Available tables:", tables)

Available tables: [('sqlite_sequence',), ('Player_Attributes',), ('Player',), ('Match',), ('League',), ('Country',), ('Team',), ('Team_Attributes',)]


In [6]:
df = pd.read_sql("SELECT * FROM Team_Attributes ", connection)
display(df)

,id,team_fifa_api_id,team_api_id,date,buildUpPlaySpeed,buildUpPlaySpeedClass,buildUpPlayDribbling,buildUpPlayDribblingClass,buildUpPlayPassing,buildUpPlayPassingClass,...,chanceCreationShooting,chanceCreationShootingClass,chanceCreationPositioningClass,defencePressure,defencePressureClass,defenceAggression,defenceAggressionClass,defenceTeamWidth,defenceTeamWidthClass,defenceDefenderLineClass
0,1,434,9930,2010-02-22 00:00:00,60,Balanced,NaN,Little,50,Mixed,...,55,Normal,Organised,50,Medium,55,Press,45,Normal,Cover
1,2,434,9930,2014-09-19 00:00:00,52,Balanced,48.0,Normal,56,Mixed,...,64,Normal,Organised,47,Medium,44,Press,54,Normal,Cover
2,3,434,9930,2015-09-10 00:00:00,47,Balanced,41.0,Normal,54,Mixed,...,64,Normal,Organised,47,Medium,44,Press,54,Normal,Cover
3,4,77,8485,2010-02-22 00:00:00,70,Fast,NaN,Little,70,Long,...,70,Lots,Organised,60,Medium,70,Double,70,Wide,Cover
4,5,77,8485,2011-02-22 00:00:00,47,Balanced,NaN,Little,52,Mixed,...,52,Normal,Organised,47,Medium,47,Press,52,Normal,Cover
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1453,1454,15005,10000,2011-02-22 00:00:00,52,Balanced,NaN,Little,52,Mixed,...,53,Normal,Organised,46,Medium,48,Press,53,Normal,Cover
1454,1455,15005,10000,2012-02-22 00:00:00,54,Balanced,NaN,Little,51,Mixed,...,50,Normal,Organised,44,Medium,55,Press,53,Normal,Cover
1455,1456,15005,10000,2013-09-20 00:00:00,54,Balanced,NaN,Little,51,Mixed,...,32,Little,Organised,44,Medium,58,Press,37,Normal,Cover
1456,1457,15005,10000,2014-09-19 00:00:00,54,Balanced,42.0,Normal,51,Mixed,...,32,Little,Organised,44,Medium,58,Press,37,Normal,Cover


In [7]:
# Load any table into a DataFrame
df = pd.read_sql("SELECT * FROM Country ", connection)
display(df)

,id,name
0,1,Belgium
1,1729,England
2,4769,France
3,7809,Germany
4,10257,Italy
5,13274,Netherlands
6,15722,Poland
7,17642,Portugal
8,19694,Scotland
9,21518,Spain


In [8]:
df = pd.read_sql("SELECT * FROM League ", connection)
display(df)

,id,country_id,name
0,1,1,Belgium Jupiler League
1,1729,1729,England Premier League
2,4769,4769,France Ligue 1
3,7809,7809,Germany 1. Bundesliga
4,10257,10257,Italy Serie A
5,13274,13274,Netherlands Eredivisie
6,15722,15722,Poland Ekstraklasa
7,17642,17642,Portugal Liga ZON Sagres
8,19694,19694,Scotland Premier League
9,21518,21518,Spain LIGA BBVA


| Columns            	| Beschrijving                                                         	| Relevantie 	|
|--------------------	|----------------------------------------------------------------------	|------------	|
| League(id)         	| Het eigen unieke nummer van de competitie.                           	| 1          	|
| League(country_id) 	| Het nummer dat naar het land verwijst.                               	| 1          	|
| League(name)       	| De naam zoals wij die kennen, bijvoorbeeld "Netherlands Eredivisie". 	| 1          	|
| Country(id)        	| is een uniek codenummer dat de computer gebruikt om te zoeken.       	| 1          	|
| Country(name)      	| De name is gewoon de naam zoals wij die kennen, zoals "Netherlands". 	| 1          	|

### Identificatie voor Sparta Rotterdam

In [13]:
import pandas as pd

data = {
    'Type Identifier': [
        'Club Naam (Lang)', 
        'Club Naam (Kort)', 
        'Team API ID', 
        'Team FIFA ID', 
        'League ID', 
        'Country ID', 
        'Team Attribute ID '
    ],
    'Waarde': [
        'Sparta Rotterdam', 
        'SPA', 
        10217, 
        1006, 
        13274, 
        13274, 
        1202
    ],
}

df_identifiers = pd.DataFrame(data)

display(df_identifiers)

,Type Identifier,Waarde
0,Club Naam (Lang),Sparta Rotterdam
1,Club Naam (Kort),SPA
2,Team API ID,10217
3,Team FIFA ID,1006
4,League ID,13274
5,Country ID,13274
6,Team Attribute ID,1202
